## 01 - Bronze Ingestion

Este notebook é responsável por ingerir os dados brutos armazenados no Volume do Unity Catalog e salvá-los no formato Delta Lake 
no schema `bronze`.

Nenhum tratamento de qualidade é aplicado nesta camada, apenas a adição de metadados (`_bronze_ingested_at`).

> **Pré-requisito**: Para rodar a ingestão de Excel, utilizamos o Pandas com a biblioteca `openpyxl` (instalada via %pip no início deste notebook).

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ./utils

# Utils

Notebook contendo funções reutilizáveis para o pipeline de dados.

In [0]:
import pyspark.sql.functions as F

In [0]:
# 1. atendimento_ocorrencias.ndjson
print("Ingerindo Ocorrencias...")
df_ocorrencias = spark.read.json(f"{source_volume_path}atendimento_ocorrencias.ndjson")
df_ocorrencias = df_ocorrencias.withColumn("_bronze_ingested_at", F.current_timestamp())
save_delta_table(df_ocorrencias, "ocorrencias", "bronze", catalog=catalog_name)

Ingerindo Ocorrencias...
Salvando tabela gerenciada: workspace.bronze.ocorrencias no modo overwrite...
Salvo com sucesso!


In [0]:
# 2. cadastro_produtos_api_dump.json (Carga Incremental com MERGE)
print("Ingerindo Produtos (Carga Incremental)...")

# Tentar obter o Watermark (maior data de atualização já processada)
watermark = "1900-01-01 00:00:00"
full_table_name = f"{catalog_name}.bronze.produtos_dump"

if spark.catalog.tableExists(full_table_name):
    watermark_val = spark.table(full_table_name).select(F.max("updated_at")).collect()[0][0]
    if watermark_val: watermark = watermark_val

print(f"Filtrando registros com updated_at > {watermark}")

df_produtos = spark.read.option("multiline","true").json(f"{source_volume_path}cadastro_produtos_api_dump.json")
df_produtos_inc = df_produtos.filter(F.col("updated_at") > watermark)

if df_produtos_inc.count() > 0:
    df_produtos_inc = df_produtos_inc.withColumn("_bronze_ingested_at", F.current_timestamp())
    # Para a Bronze, precisamos de uma chave única plana para o Merge.
    # Vamos extrair o id para facilitar o merge se ele estiver aninhado.
    df_produtos_inc = df_produtos_inc.withColumn("pk_product_id", F.col("product.product_id"))
    
    upsert_delta_table(df_produtos_inc, "produtos_dump", "bronze", join_key="pk_product_id", catalog=catalog_name)
else:
    print("Nenhum registro novo de produtos para processar.")


Ingerindo Produtos (Carga Incremental)...
Filtrando registros com updated_at > 2026-03-24T00:00:00
Nenhum registro novo de produtos para processar.


In [0]:
# 3. comercial_canais.xlsx
print("Ingerindo Canais...")
import pandas as pd
pdf_canais = pd.read_excel(f"{source_volume_path}comercial_canais.xlsx")
df_canais = spark.createDataFrame(pdf_canais.astype(str))
df_canais = df_canais.withColumn("_bronze_ingested_at", F.current_timestamp())
save_delta_table(df_canais, "comercial_canais", "bronze", catalog=catalog_name)

Ingerindo Canais...
Salvando tabela gerenciada: workspace.bronze.comercial_canais no modo overwrite...
Salvo com sucesso!


In [0]:
# 4. crm_clientes_export.xlsx
print("Ingerindo Clientes CRM...")
import pandas as pd
pdf_clientes = pd.read_excel(f"{source_volume_path}crm_clientes_export.xlsx")
df_clientes = spark.createDataFrame(pdf_clientes.astype(str))
df_clientes = df_clientes.withColumn("_bronze_ingested_at", F.current_timestamp())
save_delta_table(df_clientes, "clientes_crm", "bronze", catalog=catalog_name)

Ingerindo Clientes CRM...
Salvando tabela gerenciada: workspace.bronze.clientes_crm no modo overwrite...
Salvo com sucesso!


In [0]:
# 5. erp_pedidos_cabecalho_2025.csv (Carga Incremental com Watermark)
print("Ingerindo Pedidos (Cabecalho)...")

# Tentar obter o Watermark (maior data de atualização já processada)
watermark = "1900-01-01 00:00:00"
full_table_name = f"{catalog_name}.bronze.pedidos_cabecalho"

if spark.catalog.tableExists(full_table_name):
    watermark_val = spark.table(full_table_name).select(F.max("last_update")).collect()[0][0]
    if watermark_val: watermark = watermark_val

print(f"Filtrando registros com last_update > {watermark}")

df_pedidos_cabecalho = spark.read.csv(f"{source_volume_path}erp_pedidos_cabecalho_2025.csv", header=True, sep=";")

# Nota: Comparação de string funciona para o formato YYYY-MM-DD HH:MM:SS
df_pedidos_cabecalho_inc = df_pedidos_cabecalho.filter(F.col("last_update") > watermark)

if df_pedidos_cabecalho_inc.count() > 0:
    df_pedidos_cabecalho_inc = df_pedidos_cabecalho_inc.withColumn("_bronze_ingested_at", F.current_timestamp())
    upsert_delta_table(df_pedidos_cabecalho_inc, "pedidos_cabecalho", "bronze", join_key="order_id", catalog=catalog_name)
else:
    print("Nenhum registro novo de pedidos para processar.")


Ingerindo Pedidos (Cabecalho)...
Filtrando registros com last_update > 2026-03-16 00:00:00
Nenhum registro novo de pedidos para processar.


In [0]:
# 6. erp_pedidos_itens_2025.csv
print("Ingerindo Pedidos (Itens) - Carga Completa para auditoria de duplicados...")
df_pedidos_itens = spark.read.csv(f"{source_volume_path}erp_pedidos_itens_2025.csv", header=True, sep=",")
df_pedidos_itens = df_pedidos_itens.withColumn("_bronze_ingested_at", F.current_timestamp())

# Gravamos a base completa na Bronze (mesmo com duplicados) para processamento na Silver
save_delta_table(df_pedidos_itens, "pedidos_itens", "bronze", catalog=catalog_name, mode="overwrite")

Ingerindo Pedidos (Itens) - Carga Completa para auditoria de duplicados...
Salvando tabela gerenciada: workspace.bronze.pedidos_itens no modo overwrite...
Salvo com sucesso!


In [0]:
# 7. legado_regioes_pipe.txt
print("Ingerindo Regioes...")
df_regioes = spark.read.csv(f"{source_volume_path}legado_regioes_pipe.txt", header=True, sep="|")
df_regioes = df_regioes.withColumn("_bronze_ingested_at", F.current_timestamp())
save_delta_table(df_regioes, "regioes", "bronze", catalog=catalog_name)

Ingerindo Regioes...
Salvando tabela gerenciada: workspace.bronze.regioes no modo overwrite...
Salvo com sucesso!


In [0]:
# 8. logistica_entregas.json
print("Ingerindo Entregas...")
df_entregas = spark.read.option("multiline","true").json(f"{source_volume_path}logistica_entregas.json")
df_entregas = df_entregas.withColumn("_bronze_ingested_at", F.current_timestamp())
save_delta_table(df_entregas, "entregas", "bronze", catalog=catalog_name)

Ingerindo Entregas...
Salvando tabela gerenciada: workspace.bronze.entregas no modo overwrite...
Salvo com sucesso!


In [0]:
# 9. vendedores.csv
print("Ingerindo Vendedores...")
df_vendedores = spark.read.csv(f"{source_volume_path}vendedores.csv", header=True, sep=";")
df_vendedores = df_vendedores.withColumn("_bronze_ingested_at", F.current_timestamp())
save_delta_table(df_vendedores, "vendedores", "bronze", catalog=catalog_name)

print("Ingestão concluída com sucesso!")

Ingerindo Vendedores...
Salvando tabela gerenciada: workspace.bronze.vendedores no modo overwrite...
Salvo com sucesso!
Ingestão concluída com sucesso!
